# Judge the English↔Chinese outputs with Gemini (GEMBA-MQM with context)

The paper scores translations with a context-aware GPT-4 judge (GEMBA-MQM). This notebook uses **Gemini through Google AI Studio's free tier**
instead (different judge model, so the scores are **not comparable to the paper's GPT-4 numbers**; they are useful to compare our own systems with each other).
Score = −(10 × critical + 5 × major + 1 × minor errors): closer to 0 is better. The 1-shot example in the prompt was written for this repository.

**Settings:** Accelerator **None** (CPU only, does not use GPU quota), Internet **On**.
**API key:** get a free key at https://aistudio.google.com/apikey, then in this notebook *Add-ons → Secrets → Add secret* with label `GEMINI_API_KEY`
and tick it for this notebook. Never paste the key into a cell or into a chat.
**Input:** *Add Input → Notebook Output →* the notebook `train_eval_zh_kaggle` (it contains `run_zh/results_zh`).

In [ ]:
# 1. Code
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("cat"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "-b", "master", "https://github.com/namannamu18/context-aware-translation.git", "cat"], check=True)
os.chdir("/kaggle/working/cat")
subprocess.run(["git", "sparse-checkout", "set", "scripts"], check=True)
!pip install -q openai pandas sacrebleu

In [ ]:
# 2. Results of the training notebook (copied, because /kaggle/input is read-only) and the API key (read from Kaggle Secrets, never printed)
import glob, shutil
hits = glob.glob("/kaggle/input/**/results_zh/bmeld/test.en-zh.csv", recursive=True)
assert hits, "Add the training notebook's output as input first (Add Input -> Notebook Output)."
src = hits[0].split("/results_zh/")[0] + "/results_zh"
shutil.copytree(src, "/kaggle/working/results_zh", dirs_exist_ok=True)
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("results:", src, "| key loaded")

In [ ]:
# 3. Judge. Free tier: about 10 requests per minute, so 4 systems x ~90 messages take roughly 30-40 minutes. Re-running continues from the cache.
SYSTEMS = ["greedy-7b-ft-wo-context", "greedy-7b-ft-w-context", "mbr", "mbr-source"]    # add "greedy-7b-w-context" to judge the base model too
JUDGE_MODEL = "gemini-2.5-flash"      # a current Flash model from AI Studio (names change; check the list there if a request fails)
MAX_DOCS = None                       # e.g. 6 = only the first 6 conversations (cheaper)
for system in SYSTEMS:
    cmd = (f"python scripts/run_context_llm.py --provider gemini --judge_model {JUDGE_MODEL} --model_name gemini --lp zh --dataset bmeld --split test "
           f"--data_dir /kaggle/working/results_zh --tgt_col {system} --rpm 10 --max_workers 3 --cache_file /kaggle/working/judge_cache.jsonl"
           + (f" --max_docs {MAX_DOCS}" if MAX_DOCS else ""))
    print("###", system)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "judged" in l or "failed" in l or "unparsable" in l or "Error" in l)[-1500:])

In [ ]:
# 4. Add the scores to the results table and show it
!python scripts/consolidate_zh_results.py --root_dir /kaggle/working --out_dir /kaggle/working/results_zh --judge_only --judge_name gemini 2>&1 | grep -E "^\||merged|wrote"

The table (`results_zh/bmeld/test.en-zh.csv`, `*.summary.md`) now has an `MQM` column. **Save Version** to keep it.
If many requests failed (rate limit or a wrong model name), run cell 3 again: finished answers are not requested twice.